In [2]:
import pickle
from FlyOutput import FlyOutput
import Plotters
import plotly.graph_objects as go
import numpy as np  
import Utils
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import time
import pandas as pd
from PoseEstimation import PoseEstimation
import matplotlib.cm as cm

%matplotlib qt





path_output = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'

model_name = 'fly_model_to_fly'
file_name = 'fly_model'

dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'



path_angles = f'{path_output}/{model_name}/{file_name}_angles.pkl'
path_results = f'{path_output}/{model_name}/{file_name}_angles.pkl'


# download model_run localy
output_angles_weights_path = 'D:/Documents/gaussian_model_output/fly_model_to_fly/fly_model_results.pkl'

if os.path.exists(f'{output_angles_weights_path}'):
    with open(output_angles_weights_path, 'rb') as handle:
        output_angles_weights = pickle.load(handle)

iteration = 1200

frame0 = 1430
frame = 1710
weight_flag = False

with open(dict_path,'rb') as f:
    frames = pickle.load(f)





input_dir = 'D:/Documents/gaussian_model_output/fly_model_to_fly'


model_name = 'fly_model_to_fly'


angle_name = ['phi','theta','psi','phi','psi','yaw','pitch']
letedict = {'num_of_bins' : 20,'perc_wing_for_le' : 1, 'wing_length_snip':0.27}




with open(dict_path,'rb') as f:
    frames = pickle.load(f)
iterations = 1000

frames_fly = []
for frame in range(frame0,frame0 + 145):
    frame_output = FlyOutput(image_path,frame,input_dir,output_angles_weights,frame0,iteration,f'fly_model',letedict = letedict,deg = 0,skip_frames = 1,frames_dict = frames)
    frame_output.wings_parameters( frame_output.right_wing)
    frame_output.wings_parameters( frame_output.left_wing)
    frames_fly.append(frame_output)


In [8]:
frames_fly[0].left_wing['tip_mean']

array([ 0.01575313, -0.00531083, -0.00541338])

In [7]:
np.dot(frames_fly[0].right_wing['span'][1],frames_fly[0].left_wing['span'][1])

-0.0019895329686903033

In [6]:
np.dot(frames_fly[0].right_wing['le_ransac'][1],frames_fly[0].left_wing['le_ransac'][1])

-0.8722389386382962

In [42]:
le_ransac,le_ransac2 = [],[]
le_ransacr,le_ransac2r = [],[]

for frame in frames_fly[0:120]:
    le_ransac.append(frame.left_wing['tip_mean'] - frame.body_cm)
    # le_ransac2.append(-frame.left_wing['le_ransac'][1])


for frame in frames_fly:
    le_ransacr.append(frame.right_wing['tip_mean'] - frame.body_cm)
    # le_ransac2r.append(-frame.right_wing['le_ransac'][1])


tips  = np.vstack((le_ransac,le_ransacr))

axes = frames_fly[frame_num].get_principle_axes(tips)

In [39]:
np.linalg.norm(axes[2])

1.0000000000000002

In [43]:

frame_num  =70
rw = frames_fly[frame_num].right_wing
lw = frames_fly[frame_num].left_wing 
body = frames_fly[frame_num].body 

fig = go.Figure()
Plotters.scatter3d(fig,body - frames_fly[frame_num].body_cm,'green',4,'body',show_colorbar = False, opa=1)
# Plotters.scatter3d(fig,frame_output.right_wing,'red',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['xyz_lab'] - frames_fly[frame_num].body_cm,'cyan',4,'left wing',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,rw['xyz_lab'] - frames_fly[frame_num].body_cm,'pink',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,tips,'black',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + axes[0,:]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + axes[1,:]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack(([0,0,0],[0,0,0] + axes[2,:]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')

fig.data[-1].line.width = 15
fig.data[-2].line.width = 15
fig.data[-3].line.width = 15

fig.show()

In [12]:
fig,ax = plt.subplots(1,3)
ax[0].plot(np.vstack(le_ransac)[:,0],'*')
# ax[0].plot(np.vstack(le_ransac2)[:,0],'*')


ax[1].plot(np.vstack(le_ransac)[:,1],'*')
# ax[1].plot(np.vstack(le_ransac2)[:,1],'*')


ax[2].plot(np.vstack(le_ransac)[:,2],'*')
# ax[2].plot(np.vstack(le_ransac2)[:,2],'*')


In [26]:
fig,ax = plt.subplots(1,3)
ax[0].plot(np.vstack(le_ransacr)[:,0],'*')
ax[0].plot(np.vstack(le_ransac2r)[:,0],'*')
ax[0].plot(np.vstack(le_ransac)[:,0],'*')
ax[0].plot(np.vstack(le_ransac2)[:,0],'*')



ax[1].plot(np.vstack(le_ransacr)[:,1],'*')
ax[1].plot(np.vstack(le_ransac2r)[:,1],'*')
ax[1].plot(np.vstack(le_ransac)[:,1],'*')
ax[1].plot(np.vstack(le_ransac2)[:,1],'*')


ax[2].plot(np.vstack(le_ransacr)[:,2],'*')
ax[2].plot(np.vstack(le_ransac2r)[:,2],'*')
ax[2].plot(np.vstack(le_ransac)[:,2],'*')
ax[2].plot(np.vstack(le_ransac2)[:,2],'*')


In [21]:
fig,ax = plt.subplots(1,3)

ax[0].plot(np.vstack(le_ransac)[:,0],'*')
ax[1].plot(np.vstack(le_ransac2)[:,0],'*')

In [2]:
frame_num = 0
rw = frames_fly[frame_num].right_wing
lw = frames_fly[frame_num].left_wing

fig = go.Figure()
Plotters.scatter3d(fig,frame_output.body,'green',4,'body',show_colorbar = False, opa=1)
# Plotters.scatter3d(fig,frame_output.right_wing,'red',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['xyz_lab'],'cyan',4,'left wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['le_bins'],'blue',4,'le',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,lw['te_bins'],'gray',4,'te',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,rw['xyz_lab'],'pink',4,'right wing',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,rw['le_bins'],'red',4,'le_rw',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,rw['te_bins'],'magenta',4,'te_rw',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,np.vstack((rw['le_ransac'][0] - rw['le_ransac'][1]/1000,rw['le_ransac'][0] + rw['le_ransac'][1]/1000)),'black',20,'le ransac rw',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,np.vstack((lw['le_ransac'][0] - lw['le_ransac'][1]/1000,lw['le_ransac'][0] + lw['le_ransac'][1]/1000)),'black',20,'le ransac',show_colorbar = False, opa=1, mode = 'lines')

fig.data[-1].line.width = 15
fig.data[-2].line.width = 15

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# How many frames you want in the animation
n_frames = len(frames_fly)   # or set a smaller number if you want
def make_frame_fig(i):
    """Build a figure for a single frame index i and return its traces."""
    rw = frames_fly[i].right_wing
    lw = frames_fly[i].left_wing
    frame_output = frames_fly[i]      # <-- adjust to your actual structure

    tmp_fig = go.Figure()
    Plotters.scatter3d(tmp_fig, frame_output.body, 'green', 4, 'body',
                       show_colorbar=False, opa=1)

    Plotters.scatter3d(tmp_fig, lw['xyz_lab'], 'cyan', 4, 'left wing',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, lw['le_bins'], 'blue', 4, 'le',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, lw['te_bins'], 'gray', 4, 'te',
                       show_colorbar=False, opa=1)

    Plotters.scatter3d(tmp_fig, rw['xyz_lab'], 'pink', 4, 'right wing',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, rw['le_bins'], 'red', 4, 'le_rw',
                       show_colorbar=False, opa=1)
    Plotters.scatter3d(tmp_fig, rw['te_bins'], 'magenta', 4, 'te_rw',
                       show_colorbar=False, opa=1)

    # LE RANSAC lines (right & left)
    Plotters.scatter3d(
        tmp_fig,
        np.vstack((rw['le_ransac'][0] - rw['le_ransac'][1]/1000,
                   rw['le_ransac'][0] + rw['le_ransac'][1]/1000)),
        'black', 20, 'le ransac rw',
        show_colorbar=False, opa=1, mode='lines'
    )
    Plotters.scatter3d(
        tmp_fig,
        np.vstack((lw['le_ransac'][0] - lw['le_ransac'][1]/1000,
                   lw['le_ransac'][0] + lw['le_ransac'][1]/1000)),
        'black', 20, 'le ransac',
        show_colorbar=False, opa=1, mode='lines'
    )

    # Make the last two traces thick lines (the RANSAC lines)
    tmp_fig.data[-1].line.width = 15
    tmp_fig.data[-2].line.width = 15

    return tmp_fig.data


# --- Build base figure (frame 0) ---
fig = go.Figure()
fig.add_traces(make_frame_fig(0))

# --- Build animation frames ---
frames = []
for i in range(n_frames):
    frame_traces = make_frame_fig(i)
    frames.append(go.Frame(data=frame_traces, name=str(i)))

fig.frames = frames

# --- Slider to control frames ---
sliders = [{
    "steps": [
        {
            "method": "animate",
            "label": str(i),
            "args": [
                [str(i)],
                {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0}
                }
            ],
        }
        for i in range(n_frames)
    ],
    "transition": {"duration": 0},
    "x": 0,
    "y": 0,
    "currentvalue": {"font": {"size": 16}, "prefix": "frame: ", "visible": True, "xanchor": "right"},
    "len": 1.0,
}]

# --- Play / Pause buttons ---
updatemenus = [{
    "type": "buttons",
    "showactive": False,
    "x": 0,
    "y": 1.1,
    "buttons": [
        {
            "label": "Play",
            "method": "animate",
            "args": [
                None,
                {
                    "frame": {"duration": 50, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 0},
                },
            ],
        },
        {
            "label": "Pause",
            "method": "animate",
            "args": [
                [None],
                {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        },
    ],
}]

fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders,
    scene=dict(
        aspectmode='data'
    )
)

fig.show()


In [20]:
import plotly.graph_objects as go
import numpy as np

n_frames = len(frames_fly)   # how many frames to animate

def build_traces(i):
    rw = frames_fly[i].right_wing
    lw = frames_fly[i].left_wing
    frame_output = frames_fly[i]   # adjust if needed

    fig = go.Figure()
    Plotters.scatter3d(fig, frame_output.body, 'green', 4, 'body', opa=1)

    Plotters.scatter3d(fig, lw['xyz_lab'], 'cyan', 4, 'lw', opa=1)
    Plotters.scatter3d(fig, lw['le_bins'], 'blue', 4, 'lw_le', opa=1)
    Plotters.scatter3d(fig, lw['te_bins'], 'gray', 4, 'lw_te', opa=1)

    Plotters.scatter3d(fig, rw['xyz_lab'], 'pink', 4, 'rw', opa=1)
    Plotters.scatter3d(fig, rw['le_bins'], 'red', 4, 'rw_le', opa=1)
    Plotters.scatter3d(fig, rw['te_bins'], 'magenta', 4, 'rw_te', opa=1)

    # RANSAC lines
    Plotters.scatter3d(
        fig,
        np.vstack((rw['le_ransac'][0] - rw['le_ransac'][1]/1000,
                   rw['le_ransac'][0] + rw['le_ransac'][1]/1000)),
        'black', 20, 'rw_ransac', opa=1, mode='lines'
    )
    Plotters.scatter3d(
        fig,
        np.vstack((lw['le_ransac'][0] - lw['le_ransac'][1]/1000,
                   lw['le_ransac'][0] + lw['le_ransac'][1]/1000)),
        'black', 20, 'lw_ransac', opa=1, mode='lines'
    )

    # thicker ransac lines
    fig.data[-1].line.width = 15
    fig.data[-2].line.width = 15

    return fig.data


# --- BASE FIGURE ---
fig = go.Figure()
fig.add_traces(build_traces(0))


# --- ANIMATION FRAMES ---
frames = []
for i in range(n_frames):
    frames.append(go.Frame(data=build_traces(i), name=str(i)))

fig.frames = frames


# --- SLIDER ---
fig.update_layout(
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "Frame: "},
        "pad": {"t": 30},
        "steps": [
            {
                "args": [[str(i)], {"frame": {"duration": 0, "redraw": True},
                                    "mode": "immediate"}],
                "label": str(i),
                "method": "animate"
            }
            for i in range(n_frames)
        ]
    }]
)


# --- PLAY / PAUSE BUTTONS ---
fig.update_layout(
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.1,
        "y": 1.15,
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 50, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 0}
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }
                ]
            }
        ]
    }]
)

fig.show()
